# Deploy the VSS Orchestrator MCP and Deploy VSS from OpenClaw Within NemoClaw

Run this notebook on the same host that contains the VSS repo, will run the VSS orchestrator MCP server, and can reach the OpenClaw/NemoClaw sandbox.

This notebook walks through eight steps:

1. Checking the required prerequisites on the host.
2. Installing the NGC CLI if it is not already installed.
3. Authenticating Docker to `nvcr.io`.
4. Preparing the `services/agent/` Python environment.
5. Reviewing the orchestrator MCP configuration.
6. Referring to `tools/scripts/init_nemoclaw_vss.ipynb` for NemoClaw/OpenClaw sandbox policy setup.
7. Starting the VSS Orchestrator MCP server.
8. Using OpenClaw to initialize the MCP session and invoke the orchestrator tools to deploy VSS.

**Important:** run `tools/scripts/init_nemoclaw_vss.ipynb` first if OpenClaw/NemoClaw is not already configured on this instance.


## 1. Pre-requisites and pre-steps

Before you start this notebook, make sure these are already in place on the host:

- The repo exists on this host, typically at `/home/ubuntu/video-search-and-summarization`.
- `scripts/init_nemoclaw_vss.ipynb` has already been run successfully, or you already have a working OpenClaw/NemoClaw sandbox.
- Docker is installed and working.
- `openshell` is installed and can access the sandbox created by NemoClaw.
- `uv` and Python 3 are available so the `services/agent/` dependencies can be installed.
- If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, export `NGC_CLI_API_KEY` before running Steps 2 and 3.
- If you plan to use remote NVIDIA endpoints in profile generation, export `NVIDIA_API_KEY` before running the orchestrator tools that need it.

The next two sections are the actual pre-steps for local NIM-backed deployment:

1. Install the NGC CLI if it is missing.
2. Run `docker login nvcr.io` with `NGC_CLI_API_KEY`.

Notes:

- The orchestrator MCP server runs on the host.
- OpenClaw reaches that host-side MCP server from inside the sandbox through `http://host.openshell.internal:<port>/mcp`.
- The default port used below is `9902`.


In [ ]:
NGC_CLI_API_KEY = ""
NVIDIA_API_KEY = ""

In [ ]:
import os
import shutil
from pathlib import Path

VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", "/home/ubuntu/video-search-and-summarization")).resolve()
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
ORCHESTRATOR_DIR = AGENT_DIR / "src" / "vss_agents" / "orchestrator"
MCP_CONFIG_PATH = ORCHESTRATOR_DIR / "vss_orchestrator_mcp_config.yml"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo")
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9902"))
MCP_URL = f"http://127.0.0.1:{MCP_PORT}/mcp"
SANDBOX_MCP_URL = f"http://host.openshell.internal:{MCP_PORT}/mcp"
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("ORCHESTRATOR_DIR:", ORCHESTRATOR_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("POLICY_PATH:", POLICY_PATH)
print("NEMOCLAW_SANDBOX_NAME:", NEMOCLAW_SANDBOX_NAME)
print("MCP_URL:", MCP_URL)
print("SANDBOX_MCP_URL:", SANDBOX_MCP_URL)
print()

checks = {
    "repo directory": VSS_REPO_DIR.is_dir(),
    "agent directory": AGENT_DIR.is_dir(),
    "orchestrator directory": ORCHESTRATOR_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
    "NemoClaw policy": POLICY_PATH.is_file(),
    "python3": shutil.which("python3") is not None,
    "uv": shutil.which("uv") is not None,
    "docker": shutil.which("docker") is not None,
    "ngc": shutil.which("ngc") is not None,
    "openshell": shutil.which("openshell") is not None,
    "curl": shutil.which("curl") is not None,
}

for label, ok in checks.items():
    print(f"{'OK ' if ok else 'NO '} {label}")

print()
print("NGC_CLI_API_KEY set:", bool(os.environ.get("NGC_CLI_API_KEY")))
print("NVIDIA_API_KEY set:", bool(os.environ.get("NVIDIA_API_KEY")))


## 2. Install NGC CLI

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, the host needs the NGC CLI.

This pre-step checks whether `ngc` is already installed and installs it if needed.


In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}\n{result.stdout}")
    return result.stdout.strip()


ngc_path = shutil.which("ngc")
if ngc_path:
    version = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {version}")
else:
    arch = platform.machine()
    filename = "ngccli_linux_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    ngc_cli_version = "4.13.0"
    url = (
        "https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/"
        f"versions/{ngc_cli_version}/files/{filename}"
    )

    print(f"Installing NGC CLI {ngc_cli_version}...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")

    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(
            f"NGC CLI download failed: /tmp/ngc_cli.zip is only {size} bytes."
        )

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    version = run("ngc --version 2>&1 | head -1")
    print(f"Installed NGC CLI: {version}")


# Configure NGC CLI with API key and org
print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvstaging
""")

print("NGC CLI configured.")
print(run("ngc config current"))


## 3. Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.


In [ ]:
import os
import subprocess

ngc_cli_api_key = os.environ.get("NGC_CLI_API_KEY")
if not ngc_cli_api_key:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        ngc_cli_api_key,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")


## 4. Prepare the `services/agent/` environment

The orchestrator MCP server is part of the VSS agent package, so install the Python environment in `services/agent/` first.

This runs `uv sync` from the agent directory.


In [ ]:
import os
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError("uv is not installed. Install uv first, then re-run this cell.")

uv_env = os.environ.copy()
uv_env.pop("VIRTUAL_ENV", None)
subprocess.run(["uv", "sync"], cwd=str(AGENT_DIR), env=uv_env, check=True)
venv_info = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import os, sys; print('uv VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))",
    ],
    cwd=str(AGENT_DIR),
    env=uv_env,
    check=True,
    capture_output=True,
    text=True,
)
print(venv_info.stdout.strip())
print("Environment is ready.")


## 5. Review the orchestrator MCP configuration

The orchestrator server reads its runtime configuration from `services/agent/src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml`.

At minimum, confirm these paths are correct for this host before starting the server:

- `mdx_data_dir`
- `output_dir`
- `deployments_dir`
- `source_compose_yaml`
- `source_env`

If this repo is still under `/home/ubuntu/video-search-and-summarization`, the defaults should already be correct.


In [ ]:
import re

config_text = MCP_CONFIG_PATH.read_text()
for key in ["mdx_data_dir", "output_dir", "deployments_dir", "source_compose_yaml", "source_env"]:
    match = re.search(rf'^\s*{re.escape(key)}:\s*"?(.*?)"?\s*$', config_text, re.MULTILINE)
    print(f"{key}: {match.group(1) if match else 'NOT FOUND'}")


## 6. Use [`init_nemoclaw_vss.ipynb`](./init_nemoclaw_vss.ipynb) for sandbox policy setup

<span style="color:red"><strong>Important:</strong> run that notebook's policy configuration flow before using OpenClaw with this orchestrator MCP server.</span>

OpenClaw needs the sandbox policy updates from [`init_nemoclaw_vss.ipynb`](./init_nemoclaw_vss.ipynb) before it can reach the orchestrator MCP running on the host.

Expected sandbox endpoint after policy setup: `http://host.openshell.internal:9902/mcp`.

## 7. Start the VSS Orchestrator MCP server

This cell stops any existing MCP server tracked in notebook memory, then starts a fresh server in the background and writes logs under `.orchestrator-artifacts/`.

Default endpoint after startup: `http://127.0.0.1:9902/mcp`


In [ ]:
import os
import signal
import subprocess
import time

ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"

existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
log_handle = LOG_PATH.open("a")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        "src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml",
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
time.sleep(5)
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
print("In-memory PID:", VSS_ORCHESTRATOR_MCP_PID)
print("Log file:", LOG_PATH)
print(f"Endpoint: http://127.0.0.1:{MCP_PORT}/mcp")


In [ ]:
import json
import urllib.request

def _parse_mcp_response_body(body: str):
    data_lines = [line[len("data:") :].strip() for line in body.splitlines() if line.startswith("data:")]
    if data_lines:
        return json.loads(data_lines[-1])
    return json.loads(body)

def _mcp_post(method: str, params: dict | None = None, *, session_id: str | None = None, request_id: int = 1):
    payload = json.dumps(
        {
            "jsonrpc": "2.0",
            "method": method,
            "params": params or {},
            "id": request_id,
        }
    ).encode("utf-8")
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if session_id:
        headers["mcp-session-id"] = session_id
    req = urllib.request.Request(MCP_URL, data=payload, headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=30) as resp:
        body = resp.read().decode("utf-8")
        return resp.headers, _parse_mcp_response_body(body)

initialize_headers, initialize_body = _mcp_post(
    "initialize",
    {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "notebook", "version": "1.0"},
    },
    request_id=0,
)
session_id = initialize_headers.get("mcp-session-id")
if not session_id:
    raise RuntimeError(f"MCP initialize succeeded but no mcp-session-id header was returned: {initialize_body}")

_, tools_body = _mcp_post("tools/list", {}, session_id=session_id, request_id=1)
tool_names = [tool["name"] for tool in tools_body.get("result", {}).get("tools", [])]
print("Tool names:")
for name in tool_names:
    print(" -", name)

print()
_, profiles_body = _mcp_post(
    "tools/call",
    {"name": "vss_orchestrator__profiles", "arguments": {}},
    session_id=session_id,
    request_id=2,
)
print("profiles response:")
print(json.dumps(profiles_body, indent=2))


## 8. Use OpenClaw to initialize and call the orchestrator MCP tools

After `tools/scripts/init_nemoclaw_vss.ipynb` finishes its sandbox policy setup, the OpenClaw sandbox can reach the orchestrator MCP at:

- `http://host.openshell.internal:9902/mcp`

Use the following pattern from the OpenClaw sandbox for every MCP interaction.

### A. Minimal connectivity test from inside the OpenClaw sandbox

```bash
# Step 1: initialize and capture the session ID from the response header
SESSION_ID=$(curl -si -X POST http://host.openshell.internal:9902/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"openclaw","version":"1.0"}},"id":0}' \
  | awk 'BEGIN{IGNORECASE=1}/^mcp-session-id:/{print $2}' | tr -d '\r')

# Step 2: list tools using that session ID
curl -s -X POST http://host.openshell.internal:9902/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "mcp-session-id: $SESSION_ID" \
  -d '{"jsonrpc":"2.0","method":"tools/list","params":{},"id":1}' \
  | sed -n 's/^data: //p'
```

### B. OpenClaw prompt examples for every orchestrator MCP tool

Use these prompts in the OpenClaw UI that was configured by `tools/scripts/init_nemoclaw_vss.ipynb`.

1. Discover the tool list:
   - `Use shell access in the sandbox to initialize an MCP session against http://host.openshell.internal:9902/mcp, then call tools/list and summarize the available orchestrator tools.`

2. `profiles`:
   - `Call vss_orchestrator__profiles and tell me which deployment profiles are supported.`

3. `prereqs`:
   - `Call vss_orchestrator__prereqs and summarize any failed prerequisite checks.`

4. `docker_generate`:
   - `Call vss_orchestrator__docker_generate for profile base with env_overrides ["HARDWARE_PROFILE=RTXPRO6000BW", "HOST_IP=<host-ip>"] and return the docker_compose_id.`
   - If you are generating a local-model deployment, include your NGC key in the tool input.
   - If you are generating a remote-endpoint deployment, include your NVIDIA API key in the tool input.

5. `docker_read`:
   - `Read the generated artifacts for docker_compose_id <id> with vss_orchestrator__docker_read and summarize the resolved env and compose file.`

6. `docker_up`:
   - `Start deployment for docker_compose_id <id> with vss_orchestrator__docker_up and tell me the returned docker_compose_ops_id.`

7. `docker_status`:
   - `Poll vss_orchestrator__docker_status for docker_compose_ops_id <ops-id> until the compose operation completes, and summarize the current log excerpt.`

8. `docker_list`:
   - `Call vss_orchestrator__docker_list with all_containers=true and list the container names.`

9. `docker_logs`:
   - `Call vss_orchestrator__docker_logs for container_name <name> with tail=200 and summarize the important log lines.`

10. `docker_down`:
   - `Tear down docker_compose_id <id> with vss_orchestrator__docker_down, then keep polling vss_orchestrator__docker_status until the shutdown finishes.`

### C. Recommended operating order from OpenClaw

A typical OpenClaw flow is:

1. `profiles`
2. `prereqs`
3. `docker_generate`
4. `docker_read`
5. `docker_up`
6. `docker_status`
7. `docker_list`
8. `docker_logs`
9. `docker_down` when you want to stop the deployment

If OpenClaw cannot reach the endpoint, re-run the sandbox policy setup in `tools/scripts/init_nemoclaw_vss.ipynb` and make sure the MCP server is still listening on the same port.


In [ ]:
import os
import signal

pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if pid:
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to MCP server PID {pid}")
    except ProcessLookupError:
        print(f"Process {pid} is no longer running")
    VSS_ORCHESTRATOR_MCP_PID = None
else:
    print("No in-memory MCP server PID found. Nothing to stop.")
